# Moral foundation coding with an OpenAI model — main analysis

The baseline cell: **main prompt × 0-shot**, model `gpt-6-luna` (non-reasoning, `reasoning_effort="none"`).

Before running:
* Select the Python kernel (top right) you want to use.
* Provide your OpenAI API key (prompted below, or set `OPENAI_API_KEY` in a `.env` file — keep `.env` out of git).

## Setup

In [ ]:
# Run once, then restart the kernel
%pip install -q -U openai pandas tqdm scikit-learn ipywidgets python-dotenv

In [1]:
# Reload run_grid_openai.py automatically whenever you edit it
%load_ext autoreload
%autoreload 2

In [2]:
import os, json, getpass
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score

try:
    from dotenv import load_dotenv
    load_dotenv()                      # reads OPENAI_API_KEY from .env if present
except ImportError:
    pass
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import run_grid_openai as rg
print("model:", rg.MODEL, "| reasoning_effort:", rg.REASONING_EFFORT, "| output folder:", rg.OUT_DIR)

model: gpt-6-luna | reasoning_effort: none | output folder: runs/gpt-6-luna


## Load validation set

In [3]:
validation = rg.load_validation()
print(validation.shape)
validation.head()

(50, 20)


,para_id,moral_Yunfan,care_Yunfan,fairness_Yunfan,loyalty_Yunfan,authority_Yunfan,sanctity_Yunfan,moral_Wiktoria,care_Wiktoria,fairness_Wiktoria,loyalty_Wiktoria,authority_Wiktoria,sanctity_Wiktoria,moral_Carlos,care_Carlos,fairness_Carlos,loyalty_Carlos,authority_Carlos,sanctity_Carlos,text
0,CREC-2025-06-12-pt1-PgH2814_2_3,1,1,0,0,0,1,1,1,0,1,1,0,1,1,0,0,1,0,Every American should be outraged at how the T...
1,CREC-2025-02-07-pt1-PgH576-2_1_26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,How can this not terrify people? Are there peo...
2,CREC-2025-06-05-pt1-PgH2489_13_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,I rise today in strong opposition to H.R. 2987...
3,CREC-2025-02-25-pt1-PgH791-3_73_2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,This budget will usher in a new golden era tha...
4,CREC-2026-02-25-pt1-PgH2301-2_16_9,1,0,0,0,1,0,1,0,0,0,1,0,1,1,0,0,1,0,Even after the United States supreme_court rul...


## Prompt
The main prompt is identical to the one in `llm_moral_coding.ipynb`.

In [17]:
print(rg.make_prompt(validation["text"][0]))



1. TASK
You're an expert political scientist analyzing U.S. congressional floor speeches. You need to decide whether a paragraph from the speeches uses moral rhetoric, and if it does, which moral foundations it draws on.
Code six labels, each as 1 (present) or 0 (absent): MORAL RHETORIC, CARE, FAIRNESS, LOYALTY, AUTHORITY, SANCTITY.

2. CODING APPROACH
* The unit of analysis is the paragraph. Read the full paragraph before coding.
* Code each label for presence (1) or absence (0).
* Code a label as 1 if the paragraph expresses its meaning at least once.
* A paragraph can be coded for several moral foundations at the same time.

Procedure:
1. Read the full paragraph.
2. Code MORAL RHETORIC as 1 or 0.
3. If MORAL RHETORIC is 1, code each of the five moral foundations as 1 or 0. At least one foundation must be 1.
4. If MORAL RHETORIC is 0, all five moral foundations must be 0.

3. LABELS


Label: MORAL RHETORIC
Definition: The speaker, explicitly or implicitly,
* judges an action, actor

Test on one paragraph:

In [18]:
rg.check_params(rg.make_prompt(validation["text"][0]))   # drops temperature/seed if the model rejects them
rg.ask_llm(rg.make_prompt(validation["text"][0]), validation["text"][0])

{'moral_rhetoric': 1,
 'care': 1,
 'fairness': 1,
 'loyalty': 0,
 'authority': 1,
 'sanctity': 0,
 'evidence_moral_rhetoric': '“outraged” at the administration’s “horrible actions”',
 'evidence_care': '“ensure that my constituents, immigrants, and all Americans are protected”',
 'evidence_fairness': '“Whether you are a U.S. Senator, a U.S. Citizen, or an immigrant, no one is protected”',
 'evidence_loyalty': '',
 'evidence_authority': '“It is my job to ensure” that people “are protected”',
 'evidence_sanctity': '',
 'problem': None,
 'warnings': ['moral_rhetoric: evidence not found in paragraph',
  'authority: evidence not found in paragraph'],
 'raw': '{"evidence":{"moral_rhetoric":"“outraged” at the administration’s “horrible actions”","care":"“ensure that my constituents, immigrants, and all Americans are protected”","fairness":"“Whether you are a U.S. Senator, a U.S. Citizen, or an immigrant, no one is protected”","loyalty":"","authority":"“It is my job to ensure” that people “are 

## Run the primary cell (main × 0-shot)

Every paragraph is coded once; no resume logic. Results are saved to
`runs/gpt-6-luna/main_0shot_validation.csv`, together with the exact prompt (`_prompt.txt`)
and the settings used (`_settings.json`).

In [23]:
pred = rg.run_cell(validation, "main", 0)
pred.head()

main, 0-shot:   0%|          | 0/50 [00:00<?, ?it/s]

,para_id,text,variant,shots,moral_rhetoric,care,fairness,loyalty,authority,sanctity,evidence_moral_rhetoric,evidence_care,evidence_fairness,evidence_loyalty,evidence_authority,evidence_sanctity,problem,warnings,raw,model_version
0,CREC-2025-06-12-pt1-PgH2814_2_3,Every American should be outraged at how the T...,main,0,1,1,1,0,1,0,“outraged at how the Trump administration is t...,"“ensure that my constituents, immigrants, and ...","“Whether you are a U.S. Senator, a U.S. Citize...",,“It is my job to ensure”,,None,[moral_rhetoric: evidence not found in paragraph],"{""evidence"":{""moral_rhetoric"":""“outraged at ho...",gpt-6-luna
1,CREC-2025-02-07-pt1-PgH576-2_1_26,How can this not terrify people? Are there peo...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna
2,CREC-2025-06-05-pt1-PgH2489_13_1,I rise today in strong opposition to H.R. 2987...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna
3,CREC-2025-02-25-pt1-PgH791-3_73_2,This budget will usher in a new golden era tha...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna
4,CREC-2026-02-25-pt1-PgH2301-2_16_9,Even after the United States supreme_court rul...,main,0,1,1,0,0,1,0,"""are illegal""","""making their groceries, their appliances, and...",,,"""the President doesn't have the authority to d...",,None,[],"{""evidence"":{""moral_rhetoric"":""\""are illegal\""...",gpt-6-luna


In [20]:
settings = json.load(open(f"{rg.OUT_DIR}/main_0shot_validation_settings.json"))
print("dropped parameters:", settings["dropped_params"])     # [] = temperature and seed were used
print("model versions returned:", settings["model_versions"])
print("params:", settings["params"])

dropped parameters: []
model versions returned: ['gpt-6-luna']
params: {'model': 'gpt-6-luna', 'temperature': 0, 'seed': 502, 'reasoning_effort': 'none'}


## Output checks

In [24]:
print("problems:")
print(pred["problem"].fillna("none").value_counts())
warnings = pred["warnings"].apply(lambda w: w if isinstance(w, list) else [])
print("\nwarnings:")
print(pd.Series([x.split(": ", 1)[1] for ws in warnings for x in ws], dtype=str).value_counts())

problems:
problem
none    50
Name: count, dtype: int64

warnings:
evidence not found in paragraph    5
evidence given but coded 0         4
coded 1 without evidence           1
Name: count, dtype: int64


## Evaluation against human labels

In [25]:
EVAL_LABELS = ["moral"] + rg.FOUNDATIONS
CODERS = ["Yunfan", "Wiktoria", "Carlos"]
LLM_COL = {"moral": "moral_rhetoric", **{f: f for f in rg.FOUNDATIONS}}

m = validation.merge(pred.drop(columns=["text"], errors="ignore"), on="para_id")
m = m[m["moral_rhetoric"].notna()]          # drop parse/API errors
for l in EVAL_LABELS:                        # majority vote of the three coders
    m[f"{l}_majority"] = (m[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)

def evaluate(df, human_col_fn, llm_col_map=LLM_COL):
    rows = []
    for l in EVAL_LABELS:
        sub = df[[human_col_fn(l), llm_col_map[l]]].dropna()
        y, yhat = sub.iloc[:, 0].astype(int), sub.iloc[:, 1].astype(int)
        p, r, f, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
        rows.append({"label": l, "n": len(sub), "human_n": y.sum(), "llm_n": yhat.sum(),
                     "accuracy": (y == yhat).mean(), "precision": p, "recall": r, "f1": f,
                     "kappa": cohen_kappa_score(y, yhat)})
    return pd.DataFrame(rows).set_index("label").round(3)

all_results = {c: evaluate(m, lambda l, c=c: f"{l}_{c}") for c in CODERS}
all_results["majority"] = evaluate(m, lambda l: f"{l}_majority")
results = pd.concat(all_results, names=["coder", "label"])

print("n =", len(m))
for c, res in all_results.items():
    print(f"{c:>9} macro-F1 (5 foundations):", res.loc[rg.FOUNDATIONS, "f1"].mean().round(3))
results

n = 50
   Yunfan macro-F1 (5 foundations): 0.443
 Wiktoria macro-F1 (5 foundations): 0.325
   Carlos macro-F1 (5 foundations): 0.633
 majority macro-F1 (5 foundations): 0.529


n  human_n  llm_n  accuracy  precision  recall     f1  \
coder    label                                                               
Yunfan   moral      50       19     20      0.78      0.700   0.737  0.718   
         care       50        5      9      0.84      0.333   0.600  0.429   
         fairness   50        4      7      0.86      0.286   0.500  0.364   
         loyalty    50        4      7      0.82      0.143   0.250  0.182   
         authority  50        5     11      0.88      0.455   1.000  0.625   
         sanctity   50        7      6      0.90      0.667   0.571  0.615   
Wiktoria moral      50       21     20      0.66      0.600   0.571  0.585   
         care       50       11      9      0.76      0.444   0.364  0.400   
         fairness   50        3      7      0.84      0.143   0.333  0.200   
         loyalty    50        4      7      0.78      0.000   0.000  0.000   
         authority  50        8     11      0.82      0.455   0.625  0.526   
         sanctity   50        2      6      0.92      0.333   1.000  0.500   
Carlos   moral      50       27     20      0.82      0.950   0.704  0.809   
         care       50        9      9      0.92      0.778   0.778  0.778   
         fairness   50        6      7      0.86      0.429   0.500  0.462   
         loyalty    50        5      7      0.80      0.143   0.200  0.167   
         authority  50       12     11      0.98      1.000   0.917  0.957   
         sanctity   50        4      6      0.96      0.667   1.000  0.800   
majority moral      50       24     20      0.76      0.800   0.667  0.727   
         care       50        7      9      0.84      0.444   0.571  0.500   
         fairness   50        3      7      0.88      0.286   0.667  0.400   
         loyalty    50        5      7      0.80      0.143   0.200  0.167   
         authority  50        7     11      0.92      0.636   1.000  0.778   
         sanctity   50        4      6      0.96      0.667   1.000  0.800   

                    kappa  
coder    label             
Yunfan   moral      0.538  
         care       0.344  
         fairness   0.291  
         loyalty    0.089  
         authority  0.565  
         sanctity   0.558  
Wiktoria moral      0.298  
         care       0.252  
         fairness   0.127  
         loyalty   -0.113  
         authority  0.419  
         sanctity   0.468  
Carlos   moral      0.646  
         care       0.729  
         fairness   0.382  
         loyalty    0.057  
         authority  0.944  
         sanctity   0.779  
majority moral      0.516  
         care       0.407  
         fairness   0.345  
         loyalty    0.057  
         authority  0.732  
         sanctity   0.779

In [26]:
results.to_csv(f"{rg.OUT_DIR}/main_0shot_evaluation.csv")

### LLM–human agreement vs. human–human agreement (Cohen's κ)

Is the LLM within the range of agreement among the human coders?

In [27]:
from itertools import combinations

rows = []
for l in EVAL_LABELS:
    hh = [cohen_kappa_score(m[f"{l}_{a}"].astype(int), m[f"{l}_{b}"].astype(int))
          for a, b in combinations(CODERS, 2)]
    lh = [cohen_kappa_score(m[f"{l}_{c}"].astype(int), m[LLM_COL[l]].astype(int)) for c in CODERS]
    rows.append({"label": l,
                 "human-human mean": sum(hh) / 3, "human-human min": min(hh), "human-human max": max(hh),
                 "LLM-human mean": sum(lh) / 3, "LLM-human min": min(lh), "LLM-human max": max(lh)})
pd.DataFrame(rows).set_index("label").round(3)

,human-human mean,human-human min,human-human max,LLM-human mean,LLM-human min,LLM-human max
label,,,,,,
moral,0.438,0.368,0.529,0.494,0.298,0.646
care,0.345,0.252,0.508,0.442,0.252,0.729
fairness,0.132,-0.087,0.558,0.267,0.127,0.382
loyalty,0.322,0.146,0.634,0.011,-0.113,0.089
authority,0.429,0.381,0.521,0.642,0.419,0.944
sanctity,0.584,0.408,0.696,0.602,0.468,0.779


## Disagreements for error analysis

In [28]:
EVID_COL = {l: f"evidence_{LLM_COL[l]}" for l in EVAL_LABELS}

def find_disagreements(df, coder="majority", outfile=None):
    """coder: 'Yunfan' / 'Wiktoria' / 'Carlos' / 'majority'"""
    def diff(row):
        out = []
        for l in EVAL_LABELS:
            h, p = row[f"{l}_{coder}"], row[LLM_COL[l]]
            if pd.isna(h) or pd.isna(p):
                continue
            if int(h) != int(p):
                out.append(l)
        return out

    df = df.copy()
    df["disagree_on"] = df.apply(diff, axis=1)
    errors = df[df["disagree_on"].str.len() > 0]
    cols = (["para_id", "text", "disagree_on"]
            + [c for l in EVAL_LABELS for c in (f"{l}_{coder}", LLM_COL[l])]
            + [EVID_COL[l] for l in EVAL_LABELS])
    if outfile:
        errors[cols].to_csv(outfile, index=False)
    print(f"{coder}: {len(errors)} / {len(df)} paragraphs with at least one disagreement")
    return errors[cols]

errors = find_disagreements(m, coder="majority",
                            outfile=f"{rg.OUT_DIR}/main_0shot_disagreements_majority.csv")
errors

majority: 25 / 50 paragraphs with at least one disagreement


,para_id,text,disagree_on,moral_majority,moral_rhetoric,care_majority,care,fairness_majority,fairness,loyalty_majority,...,authority_majority,authority,sanctity_majority,sanctity,evidence_moral_rhetoric,evidence_care,evidence_fairness,evidence_loyalty,evidence_authority,evidence_sanctity
0,CREC-2025-06-12-pt1-PgH2814_2_3,Every American should be outraged at how the T...,[fairness],1,1,1,1,0,1,0,...,1,1,0,0,“outraged at how the Trump administration is t...,"“ensure that my constituents, immigrants, and ...","“Whether you are a U.S. Senator, a U.S. Citize...",,“It is my job to ensure”,
4,CREC-2026-02-25-pt1-PgH2301-2_16_9,Even after the United States supreme_court rul...,[care],1,1,0,1,0,0,0,...,1,1,0,0,"""are illegal""","""making their groceries, their appliances, and...",,,"""the President doesn't have the authority to d...",
5,CREC-2026-03-12-pt1-PgS1018_27_12,A CBS News report this week explains what is h...,[loyalty],1,1,1,1,0,0,0,...,0,0,0,0,“Families are paying the price. That is the pr...,“Families are paying the price.”,,“Democrats' political obstruction”,,
6,CREC-2025-03-31-pt2-PgS1961_185_41,We were there at the Berlin airlift. We were t...,[care],1,1,0,1,0,0,1,...,0,0,0,0,“largely indifferent to the fate of Europe”,“the fate of Europe”,,“We were there at the Berlin airlift. We were ...,,
8,CREC-2026-01-22-pt1-PgH1299_63_14,"Again, I thank her for her friendship and lead...",[moral],1,0,0,0,0,0,0,...,0,0,0,0,,,,,,
9,CREC-2025-02-07-pt1-PgH561-4_12_10,What we are about to see is Republicans previe...,"[moral, fairness]",1,0,0,0,1,0,0,...,0,0,0,0,,,,,,
11,CREC-2025-11-18-pt1-PgH4737_31_6,"Let me ask, if it is now the standard that jus...","[loyalty, authority]",1,1,0,0,1,1,0,...,0,1,0,0,“trying to block the certification of the elec...,,“just to be on a phone call with somebody esta...,“january_6 insurrectionists”,“trying to block the certification of the elec...,
12,CREC-2026-01-21-pt1-PgH1162_19_10,I will close by thanking End Citizens United a...,[loyalty],1,1,0,0,1,1,0,...,0,0,0,0,“a democracy by the people for the people”,,“get dark money out of politics”,“collectively we can do it”,,
13,CREC-2026-01-07-pt1-PgH112-7_9_16,The Biden administration regulated every nozzl...,"[moral, care]",1,0,1,0,0,0,0,...,0,0,0,0,,,,,,
14,CREC-2025-02-04-pt1-PgS574-3_1_4,I have known Pam Bondi for years from the AG w...,"[care, authority]",1,1,0,1,0,0,0,...,0,1,1,1,“She is in it for the right reasons.”,“a heart of gold”,,,“She cares about the rule of law.”,“She is in it for the right reasons.”


# Full grid: 3 prompt variants × 3 shot conditions

Needs, in the same folder:
* `fewshot_examples.csv` — the few-shot examples in order (first 2 rows = 2-shot, first 5 = 5-shot).
* `DEFINITIONS_PARAPHRASE` in `run_grid_openai.py` — reviewed paraphrased definitions.

Example paragraphs that are also in the validation set are left out of the 2- and 5-shot cells,
so all cells are compared on the paragraphs they have in common.

In [29]:
examples = rg.load_examples()
examples[["para_id"] + rg.LABELS]
print(rg.make_examples(examples.head(2)))          # check the 2-shot block

Coded examples:

Example 1:
Paragraph: "For me, my pro-life stance is deeply rooted in my Christian faith and belief that all human life is precious, and it is a gift from the almighty God. Defending the unborn has long been one of my highest priorities, and the first term I got elected to Congress, one of my first votes was cast in favor of life."
Answer: {"evidence": {"moral_rhetoric": "all human life is precious", "care": "Defending the unborn has long been one of my highest priorities", "fairness": "", "loyalty": "", "authority": "", "sanctity": "all human life is precious, and it is a gift from the almighty God"}, "moral_rhetoric": 1, "care": 1, "fairness": 0, "loyalty": 0, "authority": 0, "sanctity": 1}

Example 2:
Paragraph: "Every American should be outraged at how the Trump administration is treating anyone who dares to stand in their way. Whether you are a U.S. Senator, a U.S. Citizen, or an immigrant, no one is protected by the Trump regime. It is my job to ensure that my co

In [ ]:
# The primary cell (main x 0-shot) was already run above as `pred`, so run the other 8 cells.
# Each is saved as runs/gpt-6-luna/{variant}_{shots}shot_validation.csv
other_cells = [(v, k) for v in rg.VARIANTS for k in rg.SHOTS if (v, k) != rg.PRIMARY]
all_pred = pd.concat([pred, rg.run_grid(validation, cells=other_cells, examples=examples)],
                     ignore_index=True)
all_pred.groupby(["variant", "shots"]).size()

note: 2 example paragraph(s) left out of this cell


main, 2-shot:   0%|          | 0/48 [00:00<?, ?it/s]

note: 5 example paragraph(s) left out of this cell


main, 5-shot:   0%|          | 0/45 [00:00<?, ?it/s]

paraphrase, 0-shot:   0%|          | 0/50 [00:00<?, ?it/s]

note: 2 example paragraph(s) left out of this cell


... (5 outputs left out)

reverse, 2-shot:   0%|          | 0/48 [00:00<?, ?it/s]

note: 5 example paragraph(s) left out of this cell


reverse, 5-shot:   0%|          | 0/45 [00:00<?, ?it/s]

variant     shots
main        0        50
            2        48
            5        45
paraphrase  0        50
            2        48
            5        45
reverse     0        50
            2        48
            5        45
dtype: int64

In [31]:
for (v, k), g in all_pred.groupby(["variant", "shots"]):
    mm = validation.merge(g.drop(columns=["text"], errors="ignore"), on="para_id")
    mm = mm[mm["moral_rhetoric"].notna()]
    for l in EVAL_LABELS:
        mm[f"{l}_majority"] = (mm[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)
    res = {c: evaluate(mm, lambda l, c=c: f"{l}_{c}") for c in CODERS}
    res["majority"] = evaluate(mm, lambda l: f"{l}_majority")
    pd.concat(res, names=["coder", "label"]).to_csv(f"{rg.OUT_DIR}/{v}_{k}shot_evaluation.csv")

In [33]:
%pip install krippendorff

Note: you may need to restart the kernel to use updated packages.


In [34]:
import krippendorff
from sklearn.metrics import cohen_kappa_score

def compare_runs(file_a, file_b):
    a = pd.read_csv(file_a)
    b = pd.read_csv(file_b)
    m = a.merge(b, on="para_id", suffixes=("_a", "_b"))
    rows = []
    for l in rg.LABELS:
        sub = m[[f"{l}_a", f"{l}_b"]].dropna()          # 去掉解析失败的段落
        x, y = sub[f"{l}_a"].astype(int), sub[f"{l}_b"].astype(int)
        alpha = (krippendorff.alpha(reliability_data=[x.values, y.values], level_of_measurement="nominal")
                 if len(set(x) | set(y)) > 1 else float("nan"))
        rows.append({"label": l, "n": len(sub),
                     "a_n": int(x.sum()), "b_n": int(y.sum()),
                     "pct_agree": (x == y).mean(),
                     "kripp_alpha": alpha,
                     "cohen_kappa": cohen_kappa_score(x, y)})
    return pd.DataFrame(rows).set_index("label").round(3)

# 例：baseline vs paraphrase × 0-shot
compare_runs(f"{rg.OUT_DIR}/main_0shot_validation.csv",
             f"{rg.OUT_DIR}/paraphrase_0shot_validation.csv")

,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,50,20,21,0.98,0.959,0.959
care,50,9,10,0.98,0.936,0.935
fairness,50,7,7,0.96,0.836,0.834
loyalty,50,7,6,0.86,0.387,0.382
authority,50,11,13,0.96,0.891,0.891
sanctity,50,6,6,0.96,0.812,0.811


In [35]:


# baseline 和 8 个 alternative conditions 逐一比较（对应 Yang 的 Table 7）
for v in rg.VARIANTS:
    for k in rg.SHOTS:
        if (v, k) == rg.PRIMARY:
            continue
        print(f"main_0shot vs {v}_{k}shot")
        display(compare_runs(f"{rg.OUT_DIR}/main_0shot_validation.csv",
                             f"{rg.OUT_DIR}/{v}_{k}shot_validation.csv"))

main_0shot vs main_2shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,48,18,16,0.917,0.820,0.818
care,48,7,5,0.958,0.812,0.810
fairness,48,6,6,0.958,0.812,0.810
loyalty,48,7,7,0.833,0.338,0.331
authority,48,10,8,0.917,0.729,0.727
sanctity,48,5,5,0.958,0.779,0.777


main_0shot vs main_5shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,45,16,16,0.911,0.808,0.806
care,45,6,5,0.978,0.898,0.897
fairness,45,5,4,0.978,0.878,0.877
loyalty,45,7,8,0.889,0.604,0.600
authority,45,8,8,0.956,0.850,0.848
sanctity,45,5,5,0.956,0.778,0.775


main_0shot vs paraphrase_0shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,50,20,21,0.98,0.959,0.959
care,50,9,10,0.98,0.936,0.935
fairness,50,7,7,0.96,0.836,0.834
loyalty,50,7,6,0.86,0.387,0.382
authority,50,11,13,0.96,0.891,0.891
sanctity,50,6,6,0.96,0.812,0.811


... (6 outputs left out)

main_0shot vs reverse_2shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,48,18,21,0.854,0.701,0.699
care,48,7,10,0.938,0.788,0.787
fairness,48,6,5,0.979,0.898,0.897
loyalty,48,7,12,0.854,0.545,0.548
authority,48,10,9,0.979,0.935,0.934
sanctity,48,5,5,0.958,0.779,0.777


main_0shot vs reverse_5shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,45,16,21,0.844,0.682,0.683
care,45,6,11,0.889,0.641,0.645
fairness,45,5,6,0.933,0.693,0.690
loyalty,45,7,12,0.889,0.670,0.672
authority,45,8,9,0.978,0.928,0.928
sanctity,45,5,6,0.933,0.693,0.690


## Robustness cells (run later)

Before running these:
* **paraphrase** — reword `DEFINITIONS_PARAPHRASE` in `run_grid_openai.py` (now a copy of the main definitions).
* **2- / 5-shot** — create `fewshot_examples.csv` (columns: `para_id, text`, the six labels with 1 = present, optional `evidence_<label>`). Example paragraphs that are also in the validation set are left out of that cell.
* **reverse** — nothing to prepare; codes are flipped back to 1 = present when saved.

In [ ]:
# all_pred = rg.run_grid(validation)                                 # all 9 cells
# all_pred = rg.run_grid(validation, cells=[("reverse", 0)])         # a single robustness cell